In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/churn_feature_engineered.csv")

print("Shape:", df.shape)
df.head()

Shape: (7043, 30)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Churn,TenureGroup,AvgMonthlySpend,ServiceCount,SecuritySupportCount,IsNewCustomer,HighMonthlyCharge,IsMonthToMonth,IsElectronicCheck,HighRiskCustomer
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,0-12 Months,29.850000,1,1,1,0,1,1,1
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,25-48 Months,55.573529,3,2,0,0,0,0,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Yes,0-12 Months,54.075000,3,2,1,0,1,0,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,25-48 Months,40.905556,3,3,0,0,0,0,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Yes,0-12 Months,75.825000,1,0,1,1,1,1,1


In [2]:
import sqlite3

connection = sqlite3.connect("../data/processed/churn.db")

df.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

print("Customers table created successfully!")

Customers table created successfully!


In [3]:
query = """
SELECT *
FROM customers
LIMIT 5;
"""

result = pd.read_sql_query(query, connection)

result

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Churn,TenureGroup,AvgMonthlySpend,ServiceCount,SecuritySupportCount,IsNewCustomer,HighMonthlyCharge,IsMonthToMonth,IsElectronicCheck,HighRiskCustomer
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,0-12 Months,29.850000,1,1,1,0,1,1,1
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,25-48 Months,55.573529,3,2,0,0,0,0,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Yes,0-12 Months,54.075000,3,2,1,0,1,0,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,25-48 Months,40.905556,3,3,0,0,0,0,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Yes,0-12 Months,75.825000,1,0,1,1,1,1,1


In [4]:
query = """
SELECT COUNT(*) AS total_customers
FROM customers;
"""

pd.read_sql_query(query, connection)

,total_customers
0,7043


In [5]:
query = """
SELECT
    Churn,
    COUNT(*) AS customer_count
FROM customers
GROUP BY Churn;
"""

pd.read_sql_query(query, connection)

,Churn,customer_count
0,No,5174
1,Yes,1869


In [6]:
query = """
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate
FROM customers;
"""

pd.read_sql_query(query, connection)

,churn_rate
0,26.54


In [7]:
query = """
SELECT
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, connection)

,Contract,total_customers,churned_customers,churn_rate
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


In [8]:
query = """
SELECT
    PaymentMethod,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, connection)

,PaymentMethod,total_customers,churned_customers,churn_rate
0,Electronic check,2365,1071,45.29
1,Mailed check,1612,308,19.11
2,Bank transfer (automatic),1544,258,16.71
3,Credit card (automatic),1522,232,15.24


In [9]:
query = """
SELECT
    InternetService,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY InternetService
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, connection)

,InternetService,total_customers,churned_customers,churn_rate
0,Fiber optic,3096,1297,41.89
1,DSL,2421,459,18.96
2,No,1526,113,7.40


In [10]:
query = """
SELECT
    IsNewCustomer,
    IsMonthToMonth,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY
    IsNewCustomer,
    IsMonthToMonth
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, connection)

,IsNewCustomer,IsMonthToMonth,total_customers,churned_customers,churn_rate
0,1,1,1994,1024,51.35
1,0,1,1881,631,33.55
2,1,0,192,13,6.77
3,0,0,2976,201,6.75


In [11]:
query = """
SELECT
    InternetService,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY InternetService
ORDER BY churn_rate DESC;
"""

internet_result = pd.read_sql_query(query, connection)

internet_result

,InternetService,total_customers,churned_customers,churn_rate
0,Fiber optic,3096,1297,41.89
1,DSL,2421,459,18.96
2,No,1526,113,7.40


In [12]:
query = """
SELECT
    Contract,
    InternetService,
    PaymentMethod,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY
    Contract,
    InternetService,
    PaymentMethod
HAVING COUNT(*) >= 50
ORDER BY churn_rate DESC
LIMIT 10;
"""

top_segments = pd.read_sql_query(query, connection)

top_segments

,Contract,InternetService,PaymentMethod,total_customers,churned_customers,churn_rate
0,Month-to-month,Fiber optic,Electronic check,1307,789,60.37
1,Month-to-month,Fiber optic,Mailed check,201,102,50.75
2,Month-to-month,Fiber optic,Bank transfer (automatic),327,149,45.57
3,Month-to-month,Fiber optic,Credit card (automatic),293,122,41.64
4,Month-to-month,DSL,Electronic check,474,192,40.51
5,Month-to-month,DSL,Mailed check,367,113,30.79
6,Month-to-month,DSL,Credit card (automatic),185,50,27.03
7,One year,Fiber optic,Electronic check,196,51,26.02
8,Month-to-month,No,Mailed check,325,67,20.62
9,Month-to-month,No,Bank transfer (automatic),65,13,20.00


In [13]:
connection.close()

print("SQL database connection closed.")

SQL database connection closed.
